# Cross-Task Semantic Regression

This notebook extends the cross-task decoding paradigm to regression analysis. We train regressors to map neural activity to word embeddings and evaluate:
1. Within-task regression performance (train and test on same task)
2. Cross-task regression generalization (train on one task, test on another)
3. Ranked accuracy - whether the predicted embedding is close to the correct word

The ranked accuracy metric checks if the true word embedding is among the k-nearest neighbors to the predicted embedding in the embedding space.

In [1]:
import numpy as np
import scipy.io as sio
import mat73
import os
import sys
import pandas as pd
import collections
from utils.utils import reformat_raw, remove_number, get_channel_colors, reformat
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

print("Libraries imported successfully!")

Libraries imported successfully!


## Configuration

Set the patient folder and tasks to load. We'll use the same data loading pipeline as the cross-decoding analysis.

In [2]:
# Configuration
data_folder = 'data'
patient = 'RB'
patient_date = '2024-01-25 RB'

# Tasks to load
tasks_to_load = [
    {'file': 'pictureNaming_all_data.mat', 'task_type': 'picture_naming'},
    {'file': 'pictureFlashing_all_data.mat', 'task_type': 'picture_flashing'},
    {'file': 'auditoryNaming_all_data.mat', 'task_type': 'auditory_naming'},
    {'file': 'auditoryRepetition_all_data.mat', 'task_type': 'auditory_repetition'},
]

# Binning parameters
bin_size = 100  # ms

# Time alignment options
alignment_method = 'time_warp'  # Options: 'time_warp', 'keep_original', 'truncate'
# - 'time_warp': Warp auditory tasks to match visual task timing
# - 'keep_original': Keep original trial lengths (no truncation/warping)
# - 'truncate': Truncate all tasks to minimum length

# Time warping reference (for 'time_warp' method)
# Auditory tasks have longer stimulus presentation, so we warp them to match visual tasks
reference_task = 'picture_naming'  # Use this task's timing as reference

# Bad channels and trials (can be customized per task)
bad_channels = []
bad_trials = []

# Shank selection
shank_of_interest = ['A', 'B', 'C', 'D', 'T', 'U', 'L', 'W', 'O', 'J', 'X', 'Y', 'V', 'Z']

print(f"Patient: {patient}")
print(f"Patient folder: {patient_date}")
print(f"Tasks to load: {[t['task_type'] for t in tasks_to_load]}")
print(f"Alignment method: {alignment_method}")
if alignment_method == 'time_warp':
    print(f"Reference task for warping: {reference_task}")

Patient: RB
Patient folder: 2024-01-25 RB
Tasks to load: ['picture_naming', 'picture_flashing', 'auditory_naming', 'auditory_repetition']
Alignment method: time_warp
Reference task for warping: picture_naming


## Load Word Embeddings

Load pre-computed word embeddings (e.g., from Word2Vec, GloVe, BERT, etc.)

In [3]:
# Load GloVe word embeddings
print("Loading GloVe embeddings...")

from torchtext.vocab import GloVe
from nltk.stem import WordNetLemmatizer

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Load GloVe embeddings (this may take a moment on first run)
embed = GloVe(dim=300, name='840B')  # 300-dimensional GloVe embeddings

print(f"GloVe embeddings loaded! Vocabulary size: {len(embed.itos)}")
print("Ready to create word embeddings for target words")

Loading GloVe embeddings...


c:\Users\Owner\miniconda3\envs\Speech\lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\Owner\miniconda3\envs\Speech\lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


GloVe embeddings loaded! Vocabulary size: 2196017
Ready to create word embeddings for target words


## Load Task Data

We'll use the same data loading function from the cross-decoding notebook.

## Time Alignment Strategies

Different tasks have different trial timing characteristics:
- **Visual tasks** (picture naming, picture flashing): Short, consistent trial lengths
- **Auditory tasks** (auditory naming, auditory repetition): Longer stimulus presentation, semantic processing happens later

### Alignment Methods:

1. **`time_warp`** (Recommended for cross-task comparison):
   - Warps auditory trials to match visual task timing
   - Preserves all temporal information but compresses/expands time
   - Good for comparing neural dynamics across tasks
   - Uses interpolation to align stimulus onset â†’ voice onset periods

2. **`keep_original`**:
   - Keeps original trial lengths for each task
   - Different tasks will have different numbers of time bins
   - Better for within-task analysis where timing is critical
   - Requires separate analysis per task (no concatenation)

3. **`truncate`**:
   - Truncates all trials to minimum length across tasks
   - May lose important information from auditory tasks
   - Simple but not recommended for auditory naming where semantic processing is delayed

In [4]:
def load_task_data(task_path, task_type, bin_size=100, bad_channels=[], bad_trials=[], 
                   shank_of_interest=None, word_to_category_dict=None, apply_time_warping=False):
    """
    Load and preprocess data from a single task.
    
    Parameters:
    -----------
    task_path : str
        Path to the .mat file
    task_type : str
        Label for the task type (e.g., 'picture_naming', 'auditory_naming')
        For 'picture_naming_words_aloud', this will be split into 'picture_naming' and 'word_reading'
        based on column 13 (isPicture flag)
    bin_size : int
        Bin size in milliseconds
    bad_channels : list
        Indices of bad channels to remove
    bad_trials : list
        Indices of bad trials to remove
    shank_of_interest : list or None
        List of shank prefixes to keep (e.g., ['A', 'B', 'C'])
    word_to_category_dict : dict or None
        Mapping from words to semantic categories
    apply_time_warping : bool
        Whether to apply linear time warping (for auditory tasks)
    
    Returns:
    --------
    dict with keys:
        - clean_data_binned: neural data (trials x channels x time_bins)
        - clean_trial_onset: trial onset times
        - clean_trial_offset: trial offset times
        - clean_green_screen_onset: go cue times
        - clean_voice_onset: voice onset times
        - clean_voice_offset: voice offset times
        - clean_target_words: target words
        - clean_answered_words: answered words
        - word_categories: semantic categories
        - task_type: task label for each trial
        - channel_names: channel names
        - fs: sampling frequency
        - bin_size: bin size in ms
        - clean_first_word_onset: onset time of the first word for each trial
        - clean_last_word_offset: offset time of the last word for each trial
        - data_binned_warped: time-warped data (if apply_time_warping=True)
        - *_warped: warped timing cues (if apply_time_warping=True)
    """
    print(f"\n{'='*60}")
    print(f"Loading task: {task_type}")
    print(f"File: {task_path}")
    print(f"{'='*60}")
    
    # Load data
    try:
        all_data = np.array(mat73.loadmat(task_path)['all_data'], dtype=object)
    except:
        all_data = sio.loadmat(task_path)['all_data']
    
    # Remove None trials
    all_data = all_data[~np.array([isinstance(a, type(None)) for a in all_data[:, 0]])]
    print(f"Loaded {len(all_data)} trials")
    
    # Extract sampling frequency
    fs = int(all_data[0, 10])
    n_samples_per_bin = fs * bin_size // 1000
    
    # Check if this is the pictureNaming_wordsAloud task (has isPicture flag in column 13)
    is_words_aloud_task = 'words_aloud' in task_type.lower() or 'wordsaloud' in task_type.lower()
    trial_is_picture = None
    
    if is_words_aloud_task and all_data.shape[1] > 13:
        # Column 13 contains isPicture flag (can be 'picture'/'text' strings or 1/0 numbers)
        trial_is_picture = reformat_raw(all_data[:, 13])
        # Convert to boolean, handling both string and numeric formats
        def convert_to_bool(val):
            if isinstance(val, str):
                return val.lower() == 'picture'
            else:
                try:
                    return bool(int(float(val)))
                except:
                    return True  # Default to picture if unclear
        trial_is_picture = np.array([convert_to_bool(val) for val in trial_is_picture])
        print(f"Words Aloud task detected: {np.sum(trial_is_picture)} picture trials, {np.sum(~trial_is_picture)} word reading trials")
    
    # Process target labels
    target_label = reformat_raw(all_data[:, 6])
    
    # Remove picture numbers for flashing and auditory naming
    if 'Flashing' in task_path or 'auditoryNaming' in task_path:
        target_label = np.array([remove_number(t) for t in target_label])
    
    # Get word categories if mapping is provided
    if word_to_category_dict is not None:
        word_category = [word_to_category_dict.get(word[:].lower(), 'unknown') for word in target_label]
        # Merge categories
        word_category = [w if w != 'fruit' and w != 'food (exclude fruit)' else 'food and fruit' for w in word_category]
        word_category = [w if w != 'vehicle' else 'objects and tools' for w in word_category]
    else:
        word_category = ['unknown'] * len(target_label)
    
    print(f"Word category distribution: {collections.Counter(word_category)}")
    
    # Extract timing and data
    data = all_data[:, 0]
    trial_onset = reformat_raw(all_data[:, 1])
    green_screen_onset = reformat_raw(all_data[:, 2])
    trial_offset = reformat_raw(all_data[:, 3])
    voice_onset = reformat_raw(all_data[:, 4])
    voice_offset = reformat_raw(all_data[:, 5])
    answer_label = reformat_raw(all_data[:, 7])
    
    # Load word timing information for auditory tasks (columns 15-16 in Python indexing)
    word_onsets = None
    word_offsets = None
    if 'auditoryNaming' in task_path or 'auditoryRepetition' in task_path:
        if all_data.shape[1] > 16:
            # Column 15: word onset times (list of onsets for each word in stimulus)
            # Column 16: word offset times (list of offsets for each word in stimulus)
            word_onsets = all_data[:, 15]
            word_offsets = all_data[:, 16]
            print(f"Loaded word timing information from columns 15-16")
    
    # Get channel names
    channel_names = all_data[0, 11]
    channel_names = [cn for cn in channel_names]
    
    # Process neural data
    shortest_trial = min([d.shape[0] for d in data])
    data = np.array([np.array([d for d in dt[:shortest_trial]]) for dt in data]).swapaxes(1, 2)
    min_length = data.shape[2] // n_samples_per_bin * n_samples_per_bin
    data = data[:, :, :min_length]
    data_binned = data.reshape(data.shape[0], data.shape[1], -1, n_samples_per_bin).mean(axis=3)
    
    print(f"Data shape before cleaning: {data_binned.shape}")
    
    # Time warping (optional, mainly for auditory tasks)
    data_binned_warped = None
    warped_cues = {}
    
    if apply_time_warping and word_onsets is not None and word_offsets is not None:
        print("\nApplying linear time warping based on word boundaries...")
        
        # Calculate trial durations based on first word onset to last word offset
        trial_durations = []
        first_word_onsets = []
        last_word_offsets = []
        
        for i in range(len(data)):
            # Extract word timing for this trial
            trial_word_onsets = word_onsets[i]
            trial_word_offsets = word_offsets[i]
            
            # Handle different data formats (list, array, or scalar)
            if isinstance(trial_word_onsets, (list, np.ndarray)):
                if len(trial_word_onsets) > 0:
                    first_word_onset = float(trial_word_onsets[0])
                    last_word_offset = float(trial_word_offsets[-1])
                else:
                    # Fallback to trial onset/offset if no words
                    first_word_onset = trial_onset[i]
                    last_word_offset = trial_offset[i]
            else:
                # Single word case
                first_word_onset = float(trial_word_onsets)
                last_word_offset = float(trial_word_offsets)
            
            first_word_onsets.append(first_word_onset)
            last_word_offsets.append(last_word_offset)
            
            onset_idx = int(np.round(first_word_onset * fs))
            offset_idx = int(np.round(last_word_offset * fs))
            duration = offset_idx - onset_idx
            trial_durations.append(duration)
        
        trial_durations = np.array(trial_durations)
        median_duration = int(np.median(trial_durations))
        
        print(f"Word-based durations: min={trial_durations.min()/fs:.3f}s, max={trial_durations.max()/fs:.3f}s, median={median_duration/fs:.3f}s")
        
        # Warp each trial
        data_warped = []
        trial_onset_warped = []
        first_word_onset_warped = []
        last_word_offset_warped = []
        trial_offset_warped = []
        green_screen_onset_warped = []
        voice_onset_warped = []
        voice_offset_warped = []
        
        for i in range(len(data)):
            trial = data[i]
            onset_idx = int(np.round(first_word_onsets[i] * fs))
            offset_idx = int(np.round(last_word_offsets[i] * fs))
            
            pre_onset = trial[:, :onset_idx]
            during_trial = trial[:, onset_idx:offset_idx]
            post_offset = trial[:, offset_idx:]
            
            # Warp during-trial segment
            original_time = np.arange(during_trial.shape[1])
            warped_time = np.linspace(0, during_trial.shape[1] - 1, median_duration)
            
            during_trial_warped = np.zeros((trial.shape[0], median_duration))
            for ch in range(trial.shape[0]):
                interpolator = interp1d(original_time, during_trial[ch, :], kind='linear', fill_value='extrapolate')
                during_trial_warped[ch, :] = interpolator(warped_time)
            
            trial_warped = np.concatenate([pre_onset, during_trial_warped, post_offset], axis=1)
            data_warped.append(trial_warped)
            
            # Adjust timing cues
            trial_onset_warped.append(trial_onset[i])
            new_first_word_onset = first_word_onsets[i]
            new_last_word_offset = (onset_idx + median_duration) / fs
            first_word_onset_warped.append(new_first_word_onset)
            last_word_offset_warped.append(new_last_word_offset)
            trial_offset_warped.append(new_last_word_offset)
            
            def warp_cue(cue_time):
                cue_idx = cue_time * fs
                if cue_idx < onset_idx:
                    return cue_time
                elif cue_idx > offset_idx:
                    shift = (median_duration - (offset_idx - onset_idx)) / fs
                    return cue_time + shift
                else:
                    relative_position = (cue_idx - onset_idx) / (offset_idx - onset_idx)
                    new_cue_idx = onset_idx + relative_position * median_duration
                    return new_cue_idx / fs
            
            green_screen_onset_warped.append(warp_cue(green_screen_onset[i]))
            voice_onset_warped.append(warp_cue(voice_onset[i]))
            voice_offset_warped.append(warp_cue(voice_offset[i]))
        
        # Standardize lengths and bin
        shortest_warped = min([d.shape[1] for d in data_warped])
        data_warped = np.array([d[:, :shortest_warped] for d in data_warped])
        min_length_warped = data_warped.shape[2] // n_samples_per_bin * n_samples_per_bin
        data_warped = data_warped[:, :, :min_length_warped]
        data_binned_warped = data_warped.reshape(data_warped.shape[0], data_warped.shape[1], -1, n_samples_per_bin).mean(axis=3)
        
        warped_cues = {
            'trial_onset': np.array(trial_onset_warped),
            'first_word_onset': np.array(first_word_onset_warped),
            'last_word_offset': np.array(last_word_offset_warped),
            'trial_offset': np.array(trial_offset_warped),
            'green_screen_onset': np.array(green_screen_onset_warped),
            'voice_onset': np.array(voice_onset_warped),
            'voice_offset': np.array(voice_offset_warped)
        }
        
        print(f"Warped data shape: {data_binned_warped.shape}")
    
    # Remove bad channels and trials
    clean_data_binned = np.delete(np.delete(data_binned, bad_channels, axis=1), bad_trials, axis=0)
    clean_channel_names = np.delete(channel_names, bad_channels, axis=0)
    
    clean_trial_onset = np.delete(trial_onset, bad_trials)
    clean_trial_offset = np.delete(trial_offset, bad_trials)
    clean_green_screen_onset = np.delete(green_screen_onset, bad_trials)
    clean_voice_onset = np.delete(voice_onset, bad_trials)
    clean_voice_offset = np.delete(voice_offset, bad_trials)
    clean_target_label = np.delete(target_label, bad_trials)
    clean_answer_label = np.delete(answer_label, bad_trials)
    clean_word_category = [word_category[i] for i in range(len(word_category)) if i not in bad_trials]
    
    # Clean word timing if it exists
    clean_first_word_onset = None
    clean_last_word_offset = None
    if word_onsets is not None and word_offsets is not None:
        # Extract first and last word times for each trial
        first_word_onset_list = []
        last_word_offset_list = []
        for i in range(len(word_onsets)):
            trial_word_onsets = word_onsets[i]
            trial_word_offsets = word_offsets[i]
            
            if isinstance(trial_word_onsets, (list, np.ndarray)):
                if len(trial_word_onsets) > 0:
                    first_word_onset_list.append(float(trial_word_onsets[0]))
                    last_word_offset_list.append(float(trial_word_offsets[-1]))
                else:
                    first_word_onset_list.append(trial_onset[i])
                    last_word_offset_list.append(trial_offset[i])
            else:
                first_word_onset_list.append(float(trial_word_onsets))
                last_word_offset_list.append(float(trial_word_offsets))
        
        clean_first_word_onset = np.delete(np.array(first_word_onset_list), bad_trials)
        clean_last_word_offset = np.delete(np.array(last_word_offset_list), bad_trials)
    
    # Clean warped data if it exists
    if data_binned_warped is not None:
        clean_data_binned_warped = np.delete(np.delete(data_binned_warped, bad_channels, axis=1), bad_trials, axis=0)
        for key in warped_cues:
            warped_cues[key] = np.delete(warped_cues[key], bad_trials)
    else:
        clean_data_binned_warped = None
    
    # Filter by shank if specified
    if shank_of_interest is not None:
        shank_mask = [i for i, cn in enumerate(clean_channel_names) if cn[0] in shank_of_interest]
        clean_data_binned = clean_data_binned[:, shank_mask, :]
        clean_channel_names = [clean_channel_names[i] for i in shank_mask]
        if clean_data_binned_warped is not None:
            clean_data_binned_warped = clean_data_binned_warped[:, shank_mask, :]
    
    print(f"Data shape after cleaning: {clean_data_binned.shape}")
    print(f"Number of channels: {len(clean_channel_names)}")
    print(f"Number of trials: {clean_data_binned.shape[0]}")
    
    # Create task type labels
    # For words_aloud task, create separate labels for picture vs word reading trials
    if is_words_aloud_task and trial_is_picture is not None:
        # Remove bad trials from trial_is_picture flag
        clean_trial_is_picture = np.delete(trial_is_picture, bad_trials)
        task_labels = ['picture_naming' if is_pic else 'word_reading' for is_pic in clean_trial_is_picture]
        print(f"Task labels: {collections.Counter(task_labels)}")
    else:
        task_labels = [task_type] * clean_data_binned.shape[0]
    
    # Package results
    result = {
        'clean_data_binned': clean_data_binned,
        'clean_trial_onset': clean_trial_onset,
        'clean_trial_offset': clean_trial_offset,
        'clean_green_screen_onset': clean_green_screen_onset,
        'clean_voice_onset': clean_voice_onset,
        'clean_voice_offset': clean_voice_offset,
        'clean_target_words': clean_target_label,
        'clean_answered_words': clean_answer_label,
        'word_categories': clean_word_category,
        'task_type': task_labels,
        'channel_names': clean_channel_names,
        'fs': fs,
        'bin_size': bin_size,
        'clean_first_word_onset': clean_first_word_onset,
        'clean_last_word_offset': clean_last_word_offset
    }
    
    if clean_data_binned_warped is not None:
        result['clean_data_binned_warped'] = clean_data_binned_warped
        result.update({f'{k}_warped': v for k, v in warped_cues.items()})
    
    return result

In [5]:
def time_warp_trials(data, first_word_onsets, last_word_offsets, fs=512, bin_size=100):
    """
    Apply linear time warping to neural data to align stimulus presentation periods.
    
    Warps the period between first word onset and last word offset to match a target duration.
    Based on the approach in semantics.ipynb.
    
    Parameters:
    -----------
    data : np.ndarray
        Neural data with shape (n_trials, n_channels, n_time_bins)
    first_word_onsets : np.ndarray
        Time of first word onset for each trial (in seconds)
    last_word_offsets : np.ndarray
        Time of last word offset for each trial (in seconds)
    fs : int
        Sampling frequency (Hz)
    bin_size : int
        Bin size in milliseconds
    
    Returns:
    --------
    np.ndarray
        Time-warped data with same shape as input
    """
    from scipy.interpolate import interp1d
    
    n_trials, n_channels, n_time_bins = data.shape
    n_samples_per_bin = fs * bin_size // 1000
    
    # Calculate target duration in samples
    print(f"Time warping {n_trials} trials...")

    
    # Warp each trial
    warped_trials = []
    
    for i in range(n_trials):
        # Get original onset and offset indices
        onset_idx = int(np.round(first_word_onsets[i] * fs))
        offset_idx = int(np.round(last_word_offsets[i] * fs))

        target_duration = offset_idx - onset_idx
        
        # Reconstruct unbinned trial (approximate by repeating bin values)
        trial_unbinned = np.repeat(data[i], n_samples_per_bin, axis=1)
        
        # Split into pre-onset, during-stimulus, and post-offset segments
        pre_onset = trial_unbinned[:, :onset_idx]
        during_stimulus = trial_unbinned[:, onset_idx:offset_idx]
        post_offset = trial_unbinned[:, offset_idx:]
        
        # Warp the during-stimulus segment
        original_time = np.arange(during_stimulus.shape[1])
        warped_time = np.linspace(0, during_stimulus.shape[1] - 1, target_duration)
        
        during_stimulus_warped = np.zeros((n_channels, target_duration))
        for ch in range(n_channels):
            interpolator = interp1d(original_time, during_stimulus[ch, :], 
                                   kind='linear', fill_value='extrapolate')
            during_stimulus_warped[ch, :] = interpolator(warped_time)
        
        # Concatenate segments
        trial_warped = np.concatenate([pre_onset, during_stimulus_warped, post_offset], axis=1)
        warped_trials.append(trial_warped)
    
    # Find shortest warped trial
    shortest_warped = min([t.shape[1] for t in warped_trials])
    warped_trials = np.array([t[:, :shortest_warped] for t in warped_trials])
    
    # Re-bin the warped data
    min_length_warped = warped_trials.shape[2] // n_samples_per_bin * n_samples_per_bin
    warped_trials = warped_trials[:, :, :min_length_warped]
    warped_binned = warped_trials.reshape(n_trials, n_channels, -1, n_samples_per_bin).mean(axis=3)
    
    print(f"  Warped data shape: {warped_binned.shape}")
    
    return warped_binned

In [6]:
# Load all tasks
task_data_dict = {}

for task_info in tasks_to_load:
    task_file = task_info['file']
    task_type = task_info['task_type']
    task_path = os.path.join(data_folder, patient_date, task_file)
    
    if not os.path.exists(task_path):
        print(f"WARNING: File not found: {task_path}")
        continue
    
    try:
        task_data = load_task_data(
            task_path=task_path,
            task_type=task_type,
            bin_size=bin_size,
            bad_channels=bad_channels,
            bad_trials=bad_trials,
            shank_of_interest=shank_of_interest,
            apply_time_warping=True  # Will apply later based on alignment_method
        )
        task_data_dict[task_type] = task_data
        print(f"âœ“ Successfully loaded {task_type}")
    except Exception as e:
        print(f"âœ— Error loading {task_type}: {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"Loaded {len(task_data_dict)} tasks successfully")
print(f"Tasks: {list(task_data_dict.keys())}")
print(f"{'='*60}")


Loading task: picture_naming
File: data\2024-01-25 RB\pictureNaming_all_data.mat
Loaded 153 trials
Word category distribution: Counter({'unknown': 153})
Data shape before cleaning: (153, 89, 96)
Data shape after cleaning: (153, 89, 96)
Number of channels: 89
Number of trials: 153
âœ“ Successfully loaded picture_naming

Loading task: picture_flashing
File: data\2024-01-25 RB\pictureFlashing_all_data.mat
Loaded 1529 trials
Word category distribution: Counter({'unknown': 1529})
Data shape before cleaning: (1529, 89, 31)
Data shape after cleaning: (1529, 89, 31)
Number of channels: 89
Number of trials: 1529
âœ“ Successfully loaded picture_flashing

Loading task: auditory_naming
File: data\2024-01-25 RB\auditoryNaming_all_data.mat
Loaded 206 trials
Word category distribution: Counter({'unknown': 206})
Loaded word timing information from columns 15-16
Data shape before cleaning: (206, 89, 114)

Applying linear time warping based on word boundaries...
Word-based durations: min=2.640s, max=8.

In [7]:
# Create word embeddings for all words across tasks using GloVe
all_words = []
for task_type, task_data in task_data_dict.items():
    all_words.extend(task_data['clean_target_words'])

# Get unique words
unique_words = list(set(all_words))
print(f"Found {len(unique_words)} unique words across all tasks")

# Lemmatize words and create embeddings
word_to_embedding = {}
missing_words = []

for word in unique_words:
    if word == 'NA' or word is None:
        continue
    
    # Lemmatize the word (remove non-alpha characters and lemmatize as noun)
    lemmatized = lemmatizer.lemmatize(''.join([char for char in word if char.isalpha()]), pos='n')
    
    # Get GloVe embedding
    try:
        word_to_embedding[word] = embed[lemmatized].numpy()
    except KeyError:
        # Try original word if lemmatized version not found
        try:
            word_to_embedding[word] = embed[word].numpy()
        except KeyError:
            missing_words.append(word)
            print(f"Warning: No embedding found for '{word}' (lemmatized: '{lemmatized}')")

print(f"\nCreated embeddings for {len(word_to_embedding)}/{len(unique_words)} words")
print(f"Missing embeddings: {len(missing_words)} words")
if len(missing_words) > 0:
    print(f"Missing words: {missing_words[:10]}{'...' if len(missing_words) > 10 else ''}")

# Add embeddings to each task
for task_type, task_data in task_data_dict.items():
    embeddings = []
    for word in task_data['clean_target_words']:
        if word in word_to_embedding and word != 'NA':
            embeddings.append(word_to_embedding[word])
        else:
            embeddings.append(None)
    task_data['embeddings'] = np.array(embeddings, dtype=object)
    
    # Count valid embeddings per task
    valid_count = sum(1 for e in embeddings if e is not None)
    print(f"{task_type}: {valid_count}/{len(embeddings)} trials have valid embeddings")

print(f"\nEmbedding dimension: {list(word_to_embedding.values())[0].shape[0]}")

Found 99 unique words across all tasks

Created embeddings for 99/99 words
Missing embeddings: 0 words
picture_naming: 153/153 trials have valid embeddings
picture_flashing: 1529/1529 trials have valid embeddings
auditory_naming: 206/206 trials have valid embeddings
auditory_repetition: 105/105 trials have valid embeddings

Embedding dimension: 300


In [8]:
# Apply time alignment based on selected method
print(f"\nApplying alignment method: {alignment_method}")
print("="*60)

if alignment_method == 'time_warp':
    # Time warp auditory tasks to match reference task timing
    if reference_task not in task_data_dict:
        print(f"WARNING: Reference task '{reference_task}' not found. Using first task as reference.")
        reference_task = list(task_data_dict.keys())[0]
    
    ref_data = task_data_dict[reference_task]
    ref_onset = np.nanmean(ref_data['clean_green_screen_onset'] - ref_data['clean_trial_onset'])
    ref_voice_onset = np.nanmean(ref_data['clean_voice_onset'] - ref_data['clean_trial_onset'])
    
    print(f"Reference task: {reference_task}")
    print(f"  Mean go cue onset: {ref_onset:.3f}s")
    print(f"  Mean voice onset: {ref_voice_onset:.3f}s")
    
    for task_type, task_data in task_data_dict.items():
        if 'auditory_naming' in task_type.lower():
            print(f"\nWarping {task_type}...")
            first_word_onsets=reformat_raw(task_data['clean_first_word_onset'])
            last_word_offsets=reformat_raw(task_data['clean_last_word_offset'])
            #first_word_onsets = [o[0] for o in first_word_onsets]
            #last_word_offsets = [o[-1] for o in last_word_offsets]
            # Warp to reference timing
            warped_data = time_warp_trials(
                data=task_data['clean_data_binned'],
                first_word_onsets=first_word_onsets,
                last_word_offsets=last_word_offsets,
                fs=task_data['fs'],
                bin_size=task_data['bin_size']

            )
            
            task_data['clean_data_binned_warped'] = warped_data
            task_data['clean_data_binned_original'] = task_data['clean_data_binned'].copy()
            task_data['clean_data_binned'] = warped_data  # Use warped by default
            
            print(f"  âœ“ Warped to reference timing")
        else:
            print(f"\n{task_type}: No warping needed (visual task)")
            task_data['clean_data_binned_warped'] = None
            task_data['clean_data_binned_original'] = task_data['clean_data_binned'].copy()

elif alignment_method == 'truncate':
    # Truncate all tasks to minimum length
    min_time_bins = min([task_data['clean_data_binned'].shape[2] 
                         for task_data in task_data_dict.values()])
    min_channels = min([task_data['clean_data_binned'].shape[1] 
                        for task_data in task_data_dict.values()])
    
    print(f"Truncating to: {min_channels} channels, {min_time_bins} time bins")
    
    for task_type, task_data in task_data_dict.items():
        original_shape = task_data['clean_data_binned'].shape
        task_data['clean_data_binned_original'] = task_data['clean_data_binned'].copy()
        task_data['clean_data_binned'] = task_data['clean_data_binned'][:, :min_channels, :min_time_bins]
        print(f"  {task_type}: {original_shape} â†’ {task_data['clean_data_binned'].shape}")

elif alignment_method == 'keep_original':
    # Keep original lengths
    print("Keeping original trial lengths for each task")
    print("Note: Cross-task analysis will require per-task processing")
    
    for task_type, task_data in task_data_dict.items():
        shape = task_data['clean_data_binned'].shape
        task_data['clean_data_binned_original'] = task_data['clean_data_binned'].copy()
        print(f"  {task_type}: {shape[0]} trials, {shape[1]} channels, {shape[2]} time bins")

print("\n" + "="*60)
print("Alignment complete!")


Applying alignment method: time_warp
Reference task: picture_naming
  Mean go cue onset: 5.627s
  Mean voice onset: 6.409s

picture_naming: No warping needed (visual task)

picture_flashing: No warping needed (visual task)

Warping auditory_naming...
Time warping 206 trials...
  Warped data shape: (206, 89, 114)
  âœ“ Warped to reference timing

auditory_repetition: No warping needed (visual task)

Alignment complete!


In [9]:
# Display summary statistics for each task
summary_data = []
for task_type, task_data in task_data_dict.items():
    # Count valid embeddings
    valid_embeddings = sum([1 for emb in task_data['embeddings'] if emb is not None])
    
    # Get timing info
    mean_trial_length = np.nanmean(task_data['clean_trial_offset'] - task_data['clean_trial_onset'])
    mean_voice_onset = np.nanmean(task_data['clean_voice_onset'] - task_data['clean_trial_onset'])
    
    summary_data.append({
        'Task': task_type,
        'N Trials': task_data['clean_data_binned'].shape[0],
        'N Channels': task_data['clean_data_binned'].shape[1],
        'N Time Bins': task_data['clean_data_binned'].shape[2],
        'Valid Embeddings': valid_embeddings,
        'Mean Trial Length (s)': f"{mean_trial_length:.2f}",
        'Mean Voice Onset (s)': f"{mean_voice_onset:.2f}",
        'Is Warped': task_data['clean_data_binned_warped'] is not None
    })

summary_df = pd.DataFrame(summary_data)
print("\nTask Summary:")
print(summary_df.to_string(index=False))

# Check for task-specific characteristics
print("\nTask Characteristics:")
for task_type in task_data_dict.keys():
    if 'auditory' in task_type.lower():
        print(f"  {task_type}: Auditory task (longer stimulus presentation)")
    else:
        print(f"  {task_type}: Visual task (shorter, consistent timing)")


Task Summary:
               Task  N Trials  N Channels  N Time Bins  Valid Embeddings Mean Trial Length (s) Mean Voice Onset (s)  Is Warped
     picture_naming       153          89           96               153                  9.27                 6.41      False
   picture_flashing      1529          89           31              1529                  2.71                  nan      False
    auditory_naming       206          89          114               206                 11.83                 6.87       True
auditory_repetition       105          89           67               105                  6.38                 3.40      False

Task Characteristics:
  picture_naming: Visual task (shorter, consistent timing)
  picture_flashing: Visual task (shorter, consistent timing)
  auditory_naming: Auditory task (longer stimulus presentation)
  auditory_repetition: Auditory task (longer stimulus presentation)


C:\Users\Owner\AppData\Local\Temp\ipykernel_30972\885732260.py:9: RuntimeWarning: Mean of empty slice
  mean_voice_onset = np.nanmean(task_data['clean_voice_onset'] - task_data['clean_trial_onset'])


## Prepare Regression Data

Extract neural data and corresponding word embeddings for regression analysis.

In [10]:
# Extract task data for regression analysis
# You can select which tasks to use for within-task and cross-task analysis

# Example 1: Picture Flashing vs Picture Naming (visual tasks only)
if 'picture_flashing' in task_data_dict and 'picture_naming' in task_data_dict:
    pf_data = task_data_dict['picture_flashing']
    pn_data = task_data_dict['picture_naming']
    
    # Get neural data and embeddings (swap axes: trials x time_bins x channels)
    pf_neural = pf_data['clean_data_binned'].swapaxes(1, 2)
    pf_embeddings = np.array([emb for emb in pf_data['embeddings'] if emb is not None])
    pf_words = pf_data['clean_target_words'][[emb is not None for emb in pf_data['embeddings']]]
    
    pn_neural = pn_data['clean_data_binned'].swapaxes(1, 2)
    pn_embeddings = np.array([emb for emb in pn_data['embeddings'] if emb is not None])
    pn_words = pn_data['clean_target_words'][[emb is not None for emb in pn_data['embeddings']]]
    
    print("Visual Tasks Loaded:")
    print(f"  Picture Flashing: {pf_neural.shape}, {len(pf_embeddings)} valid embeddings")
    print(f"  Picture Naming: {pn_neural.shape}, {len(pn_embeddings)} valid embeddings")

# Example 2: Auditory Naming vs Picture Naming (cross-modality)
if 'auditory_naming' in task_data_dict and 'picture_naming' in task_data_dict:
    an_data = task_data_dict['auditory_naming']
    pn_data = task_data_dict['picture_naming']
    
    an_neural = an_data['clean_data_binned'].swapaxes(1, 2)
    an_embeddings = np.array([emb for emb in an_data['embeddings'] if emb is not None])
    an_words = an_data['clean_target_words'][[emb is not None for emb in an_data['embeddings']]]
    
    print("\nCross-Modality Tasks Loaded:")
    print(f"  Auditory Naming: {an_neural.shape}, {len(an_embeddings)} valid embeddings")
    if alignment_method == 'time_warp':
        print(f"    (Time-warped to match visual task timing)")
    print(f"  Picture Naming: {pn_neural.shape}, {len(pn_embeddings)} valid embeddings")

# Example 3: Auditory Repetition vs Auditory Naming
if 'auditory_repetition' in task_data_dict and 'auditory_naming' in task_data_dict:
    ar_data = task_data_dict['auditory_repetition']
    an_data = task_data_dict['auditory_naming']
    
    ar_neural = ar_data['clean_data_binned'].swapaxes(1, 2)
    ar_embeddings = np.array([emb for emb in ar_data['embeddings'] if emb is not None])
    ar_words = ar_data['clean_target_words'][[emb is not None for emb in ar_data['embeddings']]]
    
    print("\nAuditory Tasks Loaded:")
    print(f"  Auditory Repetition: {ar_neural.shape}, {len(ar_embeddings)} valid embeddings")
    print(f"  Auditory Naming: {an_neural.shape}, {len(an_embeddings)} valid embeddings")

print("\nData preparation complete! Ready for regression analysis.")

Visual Tasks Loaded:
  Picture Flashing: (1529, 31, 89), 1529 valid embeddings
  Picture Naming: (153, 96, 89), 153 valid embeddings

Cross-Modality Tasks Loaded:
  Auditory Naming: (206, 114, 89), 206 valid embeddings
    (Time-warped to match visual task timing)
  Picture Naming: (153, 96, 89), 153 valid embeddings

Auditory Tasks Loaded:
  Auditory Repetition: (105, 67, 89), 105 valid embeddings
  Auditory Naming: (206, 114, 89), 206 valid embeddings

Data preparation complete! Ready for regression analysis.


## Within-Task Regression

Train and test regressors within each task to establish baseline performance.

In [16]:
# Import regression libraries
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from models.model import BasicRegressor
from utils.utils import plot_accuracy_plotly

print("Regression libraries imported!")

Regression libraries imported!


In [12]:
# Regression parameters
n_bins_history = 5  # Number of bins of history to use
n_splits = 5  # Number of K-fold splits
use_kfold = False  # Use K-fold cross-validation
n_epochs = 20  # Number of random epochs

# Regressor configuration
alpha = 1.0  # Ridge regularization parameter
n_pca_components = 10  # PCA components for dimensionality reduction

# Ranked accuracy parameters
compute_ranked_accuracy = True
top_k_values = [1, 3, 5, 10, 20]  # Top-k values for ranked accuracy

print(f"Regression parameters:")
print(f"  Bins of history: {n_bins_history}")
print(f"  K-fold splits: {n_splits}")
print(f"  Epochs: {n_epochs}")
print(f"  Ridge alpha: {alpha}")
print(f"  PCA components: {n_pca_components}")
print(f"  Top-k values: {top_k_values}")

Regression parameters:
  Bins of history: 5
  K-fold splits: 5
  Epochs: 20
  Ridge alpha: 1.0
  PCA components: 10
  Top-k values: [1, 3, 5, 10, 20]


In [13]:
# Train regressor on Picture Flashing data
print("Training regressor on Picture Flashing data...")

# Setup regressor
y_reducer = PCA(n_components=n_pca_components)  # Reduce word embedding dimensionality
x_reducer = None  # Keep embeddings as is
regressor = KernelRidge(alpha=alpha, kernel='rbf')

br_pf = BasicRegressor(regressor, x_reducer=x_reducer, y_reducer=y_reducer)
br_pf.load_data(pf_neural, pf_embeddings, n_bins_history=n_bins_history)
br_pf.fit(
    n_epochs=n_epochs, 
    parallel=None, 
    closest='l2',
    use_kfold=use_kfold, 
    n_splits=n_splits,
    compute_top_k_accuracy=compute_ranked_accuracy,
    top_k_values=top_k_values
)

print(f"âœ“ Picture Flashing regressor trained!")
print(f"  Results shape: {br_pf.all_test_score.shape} (epochs x time_bins)")
print(f"  Top-1 accuracy shape: {br_pf.all_top_k_accuracy[1].shape}")

Training regressor on Picture Flashing data...
âœ“ Picture Flashing regressor trained!
  Results shape: (20, 31) (epochs x time_bins)
  Top-1 accuracy shape: (20, 31)


In [14]:
# Train regressor on Picture Naming data
print("Training regressor on Picture Naming data...")

y_reducer_pn = PCA(n_components=n_pca_components)
x_reducer_pn = None
regressor_pn = Ridge(alpha=alpha)

br_pn = BasicRegressor(regressor_pn, x_reducer=x_reducer_pn, y_reducer=y_reducer_pn)
br_pn.load_data(pn_neural, pn_embeddings, n_bins_history=n_bins_history)
br_pn.fit(
    n_epochs=n_epochs, 
    parallel=None, 
    closest='l2',
    use_kfold=use_kfold, 
    n_splits=n_splits,
    compute_top_k_accuracy=compute_ranked_accuracy,
    top_k_values=top_k_values
)

print(f"âœ“ Picture Naming regressor trained!")

print("Training code (uncomment and run with actual data)")

Training regressor on Picture Naming data...
âœ“ Picture Naming regressor trained!
Training code (uncomment and run with actual data)


## Cross-Task Regression

Train regressor on Picture Flashing and test on Picture Naming to evaluate cross-task generalization.

## Cross-Modality Analysis: Auditory â†’ Visual

Test whether representations learned from auditory tasks (auditory naming/repetition) generalize to visual tasks (picture naming/flashing), and vice versa.

This is particularly interesting because:
1. Auditory tasks have longer stimulus presentation periods
2. Semantic processing may occur at different times
3. Time warping allows us to compare neural dynamics across modalities

In [ ]:
# Step 1: Visualize within-task performance from trained models
print("="*80)
print("WITHIN-TASK PERFORMANCE ANALYSIS")
print("="*80)

# Extract within-task results
pf_r2_mean = br_pf.all_test_score.mean(0)
pf_r2_std = br_pf.all_test_score.std(0)
pf_top_k_mean = {k: br_pf.all_top_k_accuracy[k].mean(0) for k in top_k_values}
pf_top_k_std = {k: br_pf.all_top_k_accuracy[k].std(0) for k in top_k_values}

pn_r2_mean = br_pn.all_test_score.mean(0)
pn_r2_std = br_pn.all_test_score.std(0)
pn_top_k_mean = {k: br_pn.all_top_k_accuracy[k].mean(0) for k in top_k_values}
pn_top_k_std = {k: br_pn.all_top_k_accuracy[k].std(0) for k in top_k_values}

# Find peak performance time bins
pf_peak_bin_r2 = np.argmax(pf_r2_mean)
pf_peak_bin_top1 = np.argmax(pf_top_k_mean[1])
pf_peak_r2 = pf_r2_mean[pf_peak_bin_r2]
pf_peak_top1 = pf_top_k_mean[1][pf_peak_bin_top1]

pn_peak_bin_r2 = np.argmax(pn_r2_mean)
pn_peak_bin_top1 = np.argmax(pn_top_k_mean[1])
pn_peak_r2 = pn_r2_mean[pn_peak_bin_r2]
pn_peak_top1 = pn_top_k_mean[1][pn_peak_bin_top1]

print(f"\nPicture Flashing (PF):")
print(f"  Peak RÂ² = {pf_peak_r2:.3f} at time bin {pf_peak_bin_r2} ({pf_peak_bin_r2*bin_size/1000:.2f}s)")
print(f"  Peak Top-1 = {pf_peak_top1:.3f} at time bin {pf_peak_bin_top1} ({pf_peak_bin_top1*bin_size/1000:.2f}s)")

print(f"\nPicture Naming (PN):")
print(f"  Peak RÂ² = {pn_peak_r2:.3f} at time bin {pn_peak_bin_r2} ({pn_peak_bin_r2*bin_size/1000:.2f}s)")
print(f"  Peak Top-1 = {pn_peak_top1:.3f} at time bin {pn_peak_bin_top1} ({pn_peak_bin_top1*bin_size/1000:.2f}s)")

# Use RÂ² peak for cross-task analysis (can be changed to top-1 if preferred)
print(f"\nâ†’ Using RÂ² peak time bins for cross-task regression")
print(f"  PF peak time bin: {pf_peak_bin_r2}")
print(f"  PN peak time bin: {pn_peak_bin_r2}")

# Step 2: Prepare all task data for cross-task testing
print("\n" + "="*80)
print("PREPARING DATA FOR CROSS-TASK TESTING")
print("="*80)

task_test_data = {}

# Picture Flashing
if 'picture_flashing' in task_data_dict:
    pf_data = task_data_dict['picture_flashing']
    pf_neural = pf_data['clean_data_binned'].swapaxes(1, 2)
    pf_embeddings = np.array([emb for emb in pf_data['embeddings'] if emb is not None])
    task_test_data['Picture Flashing'] = {
        'neural': pf_neural,
        'embeddings': pf_embeddings,
        'X_to_use': reformat(pf_neural, n_bins_history)
    }

# Picture Naming
if 'picture_naming' in task_data_dict:
    pn_data = task_data_dict['picture_naming']
    pn_neural = pn_data['clean_data_binned'].swapaxes(1, 2)
    pn_embeddings = np.array([emb for emb in pn_data['embeddings'] if emb is not None])
    task_test_data['Picture Naming'] = {
        'neural': pn_neural,
        'embeddings': pn_embeddings,
        'X_to_use': reformat(pn_neural, n_bins_history)
    }

# Auditory Naming
if 'auditory_naming' in task_data_dict:
    an_data = task_data_dict['auditory_naming']
    an_neural = an_data['clean_data_binned'].swapaxes(1, 2)
    an_embeddings = np.array([emb for emb in an_data['embeddings'] if emb is not None])
    task_test_data['Auditory Naming'] = {
        'neural': an_neural,
        'embeddings': an_embeddings,
        'X_to_use': reformat(an_neural, n_bins_history)
    }

# Auditory Repetition
if 'auditory_repetition' in task_data_dict:
    ar_data = task_data_dict['auditory_repetition']
    ar_neural = ar_data['clean_data_binned'].swapaxes(1, 2)
    ar_embeddings = np.array([emb for emb in ar_data['embeddings'] if emb is not None])
    task_test_data['Auditory Repetition'] = {
        'neural': an_neural,
        'embeddings': ar_embeddings,
        'X_to_use': reformat(ar_neural, n_bins_history)
    }

print(f"\nAvailable tasks: {list(task_test_data.keys())}")

# Step 3: Train peak-time regressors
print("\n" + "="*80)
print("TRAINING PEAK-TIME REGRESSORS")
print("="*80)

from sklearn.model_selection import KFold

def train_peak_regressor(X_train_all_bins, y_train, peak_bin, n_epochs_train=20):
    """
    Train a regressor at the peak performance time bin.
    Returns a list of trained regressors (one per epoch for robustness).
    """
    X_peak = X_train_all_bins[peak_bin]
    trained_regressors = []
    trained_reducers = []
    
    for epoch in range(n_epochs_train):
        # Apply dimensionality reduction
        reducer = PCA(n_components=n_pca_components)
        X_peak_low = reducer.fit_transform(X_peak)
        
        # Train regressor
        reg = Ridge(alpha=alpha)
        reg.fit(X_peak_low, y_train)
        
        trained_regressors.append(reg)
        trained_reducers.append(reducer)
    
    return trained_regressors, trained_reducers

# Train PF peak regressor
print(f"\nTraining PF peak regressor at time bin {pf_peak_bin_r2}...")
pf_peak_regressors, pf_peak_reducers = train_peak_regressor(
    br_pf.X_to_use, 
    pf_embeddings, 
    pf_peak_bin_r2,
    n_epochs_train=n_epochs
)
print(f"âœ“ Trained {len(pf_peak_regressors)} PF peak regressors")

# Train PN peak regressor
print(f"\nTraining PN peak regressor at time bin {pn_peak_bin_r2}...")
pn_peak_regressors, pn_peak_reducers = train_peak_regressor(
    br_pn.X_to_use, 
    pn_embeddings, 
    pn_peak_bin_r2,
    n_epochs_train=n_epochs
)
print(f"âœ“ Trained {len(pn_peak_regressors)} PN peak regressors")

# Step 4: Test peak regressors on all tasks across all time bins
print("\n" + "="*80)
print("CROSS-TASK TESTING WITH PEAK REGRESSORS")
print("="*80)

def test_peak_regressor_on_task(trained_regressors, trained_reducers, test_X_all_bins, 
                                 test_embeddings, task_name, train_task_name):
    """
    Test a peak-time regressor on all time bins of a test task.
    Handles padding for early bins where history is insufficient.
    """
    n_test_bins = len(test_X_all_bins)
    r2_scores = np.zeros(n_test_bins)
    top_k_accuracies = {k: np.zeros(n_test_bins) for k in top_k_values}
    
    print(f"\n  Testing {train_task_name} peak regressor on {task_name}...")
    
    for t in range(n_test_bins):
        if t % 20 == 0:
            print(f"    Time bin {t}/{n_test_bins}", end='\r')
        
        # Get test data at this time bin
        # For early bins (< n_bins_history), the X_to_use may have padded data
        X_test = test_X_all_bins[t]
        y_test = test_embeddings
        
        # Test with all trained regressors (epochs) and average
        epoch_r2 = []
        epoch_top_k = {k: [] for k in top_k_values}
        
        for reg, reducer in zip(trained_regressors, trained_reducers):
            # Apply same dimensionality reduction
            X_test_low = reducer.transform(X_test)
            
            # Predict and score
            y_pred = reg.predict(X_test_low)
            r2 = reg.score(X_test_low, y_test)
            epoch_r2.append(r2)
            
            # Compute top-k accuracy
            for k in top_k_values:
                n_correct = 0
                for i, pred in enumerate(y_pred):
                    distances = np.sum((test_embeddings - pred) ** 2, axis=1)
                    sorted_indices = np.argsort(distances)
                    if i in sorted_indices[:k]:
                        n_correct += 1
                epoch_top_k[k].append(n_correct / len(y_pred))
        
        r2_scores[t] = np.mean(epoch_r2)
        for k in top_k_values:
            top_k_accuracies[k][t] = np.mean(epoch_top_k[k])
    
    print(f"    Time bin {n_test_bins}/{n_test_bins} - Complete!")
    
    return {
        'r2_scores': r2_scores,
        'top_k_accuracies': top_k_accuracies,
        'n_bins': n_test_bins
    }

# Store results
cross_task_results = {}

# Store within-task performance (from trained models)
cross_task_results['PFâ†’PF (within)'] = {
    'r2_scores': pf_r2_mean,
    'r2_std': pf_r2_std,
    'top_k_accuracies': pf_top_k_mean,
    'top_k_std': pf_top_k_std,
    'n_bins': len(pf_r2_mean),
    'peak_bin': pf_peak_bin_r2
}

cross_task_results['PNâ†’PN (within)'] = {
    'r2_scores': pn_r2_mean,
    'r2_std': pn_r2_std,
    'top_k_accuracies': pn_top_k_mean,
    'top_k_std': pn_top_k_std,
    'n_bins': len(pn_r2_mean),
    'peak_bin': pn_peak_bin_r2
}

# Test PF peak regressor on all tasks
for task_name, task_data in task_test_data.items():
    result = test_peak_regressor_on_task(
        pf_peak_regressors,
        pf_peak_reducers,
        task_data['X_to_use'],
        task_data['embeddings'],
        task_name,
        'PF'
    )
    cross_task_results[f'PFâ†’{task_name}'] = result
    print(f"  â†’ Peak RÂ² = {result['r2_scores'].max():.3f}, Peak Top-1 = {result['top_k_accuracies'][1].max():.3f}")

# Test PN peak regressor on all tasks
for task_name, task_data in task_test_data.items():
    result = test_peak_regressor_on_task(
        pn_peak_regressors,
        pn_peak_reducers,
        task_data['X_to_use'],
        task_data['embeddings'],
        task_name,
        'PN'
    )
    cross_task_results[f'PNâ†’{task_name}'] = result
    print(f"  â†’ Peak RÂ² = {result['r2_scores'].max():.3f}, Peak Top-1 = {result['top_k_accuracies'][1].max():.3f}")

print("\n" + "="*80)
print("Cross-task regression analysis complete!")
print("="*80)

CROSS-TASK REGRESSION ANALYSIS

Available tasks for testing: ['Picture Flashing', 'Picture Naming', 'Auditory Naming', 'Auditory Repetition']

Trained models:
  - br_pf (Picture Flashing): 20 epochs, 31 time bins
  - br_pn (Picture Naming): 20 epochs, 96 time bins

Testing br_pf (trained on Picture Flashing) on all tasks:
âœ“ PFâ†’PF (within-task): Using stored results

Testing on Picture Naming...


KeyboardInterrupt: 

## Comparison: Time Warping vs Original Timing

For auditory tasks, compare results with and without time warping to understand the impact of alignment.

In [16]:
# Compare warped vs original timing for auditory tasks
if alignment_method == 'time_warp':
    print("Comparing time-warped vs original timing for auditory tasks...")
    print("="*60)
    
    for task_type, task_data in task_data_dict.items():
        if 'auditory' in task_type.lower() and task_data['clean_data_binned_original'] is not None:
            warped_shape = task_data['clean_data_binned'].shape
            original_shape = task_data['clean_data_binned_original'].shape
            
            print(f"\n{task_type}:")
            print(f"  Original: {original_shape}")
            print(f"  Warped:   {warped_shape}")
            
            if warped_shape[2] == original_shape[2]:
                print(f"  Status: Same length (warping preserves time bins)")
            else:
                print(f"  Status: Different lengths")
            
            # You can train regressors on both versions and compare
            # Example:
            # br_original = BasicRegressor(...)
            # br_original.load_data(original_data, embeddings, ...)
            # br_warped = BasicRegressor(...)
            # br_warped.load_data(warped_data, embeddings, ...)
            # Compare br_original.all_test_score vs br_warped.all_test_score
    
    print("\n" + "="*60)
    print("Recommendation:")
    print("  - Time-warped: Better for cross-task comparison")
    print("  - Original: Better for understanding native task dynamics")
    print("  - Train both and compare to validate findings")
else:
    print(f"Alignment method is '{alignment_method}', no time warping comparison available")

Comparing time-warped vs original timing for auditory tasks...

auditory_naming:
  Original: (206, 89, 114)
  Warped:   (206, 89, 114)
  Status: Same length (warping preserves time bins)

auditory_repetition:
  Original: (105, 89, 67)
  Warped:   (105, 89, 67)
  Status: Same length (warping preserves time bins)

Recommendation:
  - Time-warped: Better for cross-task comparison
  - Original: Better for understanding native task dynamics
  - Train both and compare to validate findings


In [17]:
# Cross-task evaluation: Train on PF, test on PN
print("Performing cross-task regression (PF train â†’ PN test) with K-fold...")

# Reformat PN data to match time bins
pn_X_to_use = reformat(pn_neural, n_bins_history)
n_effective_bins = len(br_pf.X_to_use)

# Storage for cross-task scores and ranked accuracies
from sklearn.model_selection import StratifiedKFold, KFold
pf_train_pn_test_scores = np.zeros(n_effective_bins)
pf_train_pn_test_scores_std = np.zeros(n_effective_bins)
pf_train_pn_test_ranked_acc = np.zeros(n_effective_bins)
pf_train_pn_test_top_k_acc = {k: np.zeros(n_effective_bins) for k in top_k_values}

for t in range(n_effective_bins):
    if t % 10 == 0:
        print(f"  Cross-task evaluation: time bin {t}/{n_effective_bins}")
    
    X_pf_t = br_pf.X_to_use[t]
    X_pn_t = pn_X_to_use[t]
    y_pf_t = pf_embeddings
    y_pn_t = pn_embeddings
    
    # K-fold cross-validation on PF, test on PN
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores = []
    fold_ranked_accs = []
    fold_top_k_accs = {k: [] for k in top_k_values}
    
    for train_idx, _ in kf.split(X_pf_t):
        X_train = X_pf_t[train_idx]
        y_train = y_pf_t[train_idx]
        
        # Apply dimensionality reduction
        if br_pf.x_reducer is not None:
            pca = PCA(n_components=n_pca_components)
            X_train_low = pca.fit_transform(X_train)
            X_pn_low = pca.transform(X_pn_t)
        else:
            scaler = StandardScaler()
            X_train_low = scaler.fit_transform(X_train)
            X_pn_low = scaler.transform(X_pn_t)
        
        # Train regressor
        reg = Ridge(alpha=alpha)
        reg.fit(X_train_low, y_train)
        
        # Test on PN
        y_pn_pred = reg.predict(X_pn_low)
        score = reg.score(X_pn_low, y_pn_t)
        fold_scores.append(score)
        
        # Compute ranked accuracy
        # Find closest embeddings to predictions
        n_correct_rank1 = 0
        top_k_correct = {k: 0 for k in top_k_values}
        
        for i, pred in enumerate(y_pn_pred):
            # Compute distances to all PN embeddings
            distances = np.sum((pn_embeddings - pred) ** 2, axis=1)
            sorted_indices = np.argsort(distances)
            
            # Check if true embedding is rank 1
            if sorted_indices[0] == i:
                n_correct_rank1 += 1
            
            # Check top-k
            for k in top_k_values:
                if i in sorted_indices[:k]:
                    top_k_correct[k] += 1
        
        fold_ranked_accs.append(n_correct_rank1 / len(y_pn_pred))
        for k in top_k_values:
            fold_top_k_accs[k].append(top_k_correct[k] / len(y_pn_pred))
    
    pf_train_pn_test_scores[t] = np.mean(fold_scores)
    pf_train_pn_test_scores_std[t] = np.std(fold_scores)
    pf_train_pn_test_ranked_acc[t] = np.mean(fold_ranked_accs)
    for k in top_k_values:
        pf_train_pn_test_top_k_acc[k][t] = np.mean(fold_top_k_accs[k])

print(f"âœ“ Cross-task regression complete!")
print(f"  Peak cross-task RÂ² score: {pf_train_pn_test_scores.max():.3f}")
print(f"  Peak cross-task ranked accuracy (top-1): {pf_train_pn_test_ranked_acc.max():.3f}")
print(f"  Peak cross-task top-5 accuracy: {pf_train_pn_test_top_k_acc[5].max():.3f}")

print("Cross-task regression code (uncomment and run with actual data)")

Performing cross-task regression (PF train â†’ PN test) with K-fold...
  Cross-task evaluation: time bin 0/31
  Cross-task evaluation: time bin 10/31
  Cross-task evaluation: time bin 20/31
  Cross-task evaluation: time bin 30/31
âœ“ Cross-task regression complete!
  Peak cross-task RÂ² score: -0.146
  Peak cross-task ranked accuracy (top-1): 0.033
  Peak cross-task top-5 accuracy: 0.131
Cross-task regression code (uncomment and run with actual data)


## Visualization: RÂ² Scores Over Time

Compare RÂ² scores for all cross-task combinations using plot_accuracy_plotly.

In [ ]:
# Prepare data for plot_accuracy_plotly
print("Creating RÂ² score visualization...")

# Determine common time bins
min_n_bins = min([result['n_bins'] for result in cross_task_results.values()])
print(f"Using {min_n_bins} time bins for visualization")

# Organize data by train task
# Within-task results (main data)
pf_within_r2 = cross_task_results['PFâ†’PF (within)']['r2_scores'][:min_n_bins]
pf_within_r2_std = cross_task_results['PFâ†’PF (within)']['r2_std'][:min_n_bins]

pn_within_r2 = cross_task_results['PNâ†’PN (within)']['r2_scores'][:min_n_bins]
pn_within_r2_std = cross_task_results['PNâ†’PN (within)']['r2_std'][:min_n_bins]

# Cross-task results (extra data)
pf_cross_task_data = []
pf_cross_task_labels = []
for key in ['PFâ†’Picture Naming', 'PFâ†’Auditory Naming', 'PFâ†’Auditory Repetition']:
    if key in cross_task_results:
        pf_cross_task_data.append(cross_task_results[key]['r2_scores'][:min_n_bins])
        pf_cross_task_labels.append(key)

pn_cross_task_data = []
pn_cross_task_labels = []
for key in ['PNâ†’Picture Flashing', 'PNâ†’Auditory Naming', 'PNâ†’Auditory Repetition']:
    if key in cross_task_results:
        pn_cross_task_data.append(cross_task_results[key]['r2_scores'][:min_n_bins])
        pn_cross_task_labels.append(key)

# Timing parameters
back = 0.0
forward = min_n_bins * bin_size / 1000

# Plot PF-trained regressors
print("\nPlotting PF-trained regressors...")
fig_pf_r2, ax_pf_r2 = plot_accuracy_plotly(
    pf_within_r2,
    *pf_cross_task_data,
    data_std=[pf_within_r2_std] + [None] * len(pf_cross_task_data),
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PF Peak Regressor: RÂ² Scores on All Tasks",
    ylabel="RÂ² Score",
    data_labels=['PFâ†’PF (within-task)'] + pf_cross_task_labels
)
fig_pf_r2.show()

# Plot PN-trained regressors
print("Plotting PN-trained regressors...")
fig_pn_r2, ax_pn_r2 = plot_accuracy_plotly(
    pn_within_r2,
    *pn_cross_task_data,
    data_std=[pn_within_r2_std] + [None] * len(pn_cross_task_data),
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PN Peak Regressor: RÂ² Scores on All Tasks",
    ylabel="RÂ² Score",
    data_labels=['PNâ†’PN (within-task)'] + pn_cross_task_labels
)
fig_pn_r2.show()

# Print peak RÂ² scores
print("\n" + "="*80)
print("Peak RÂ² Scores:")
print("="*80)
for combo_name in sorted(cross_task_results.keys()):
    result = cross_task_results[combo_name]
    peak_r2 = result['r2_scores'].max()
    peak_idx = result['r2_scores'].argmax()
    peak_time = peak_idx * bin_size / 1000
    marker = " â˜…" if 'peak_bin' in result else ""
    print(f"{combo_name:35s}: RÂ² = {peak_r2:.3f} at t = {peak_time:.2f}s{marker}")
print("="*80)
print("â˜… = Regressor trained at this time bin")
        bgcolor="rgba(255,255,255,0.8)",
        font=dict(size=10)
    ),
    template="plotly_white"
)

fig_r2.show()

# Print peak RÂ² scores
print("\n" + "="*80)
print("Peak RÂ² Scores:")
print("="*80)
for combo_name, result in sorted(cross_task_results.items()):
    peak_r2 = result['r2_scores'].max()
    peak_idx = result['r2_scores'].argmax()
    peak_time = time_axis[peak_idx] if peak_idx < len(time_axis) else peak_idx * bin_size / 1000
    marker = " â˜…" if 'peak_bin' in result else ""
    print(f"{combo_name:35s}: RÂ² = {peak_r2:.3f} at t = {peak_time:.2f}s{marker}")
print("="*80)
print("â˜… = Regressor trained at this time bin")

Data lengths after truncation: 31 time bins


## Visualization: Top-K Accuracy Over Time

Show top-k ranked accuracy for all cross-task combinations using plot_accuracy_plotly.

In [18]:
# Prepare Top-1 accuracy data for plot_accuracy_plotly
print("Creating Top-1 accuracy visualization...")

# PF-trained regressors - Top-1 accuracy
pf_within_top1 = cross_task_results['PFâ†’PF (within)']['top_k_accuracies'][1][:min_n_bins]
pf_within_top1_std = cross_task_results['PFâ†’PF (within)']['top_k_std'][1][:min_n_bins]

pf_cross_task_top1 = []
pf_cross_task_top1_labels = []
for key in ['PFâ†’Picture Naming', 'PFâ†’Auditory Naming', 'PFâ†’Auditory Repetition']:
    if key in cross_task_results:
        pf_cross_task_top1.append(cross_task_results[key]['top_k_accuracies'][1][:min_n_bins])
        pf_cross_task_top1_labels.append(key)

# PN-trained regressors - Top-1 accuracy
pn_within_top1 = cross_task_results['PNâ†’PN (within)']['top_k_accuracies'][1][:min_n_bins]
pn_within_top1_std = cross_task_results['PNâ†’PN (within)']['top_k_std'][1][:min_n_bins]

pn_cross_task_top1 = []
pn_cross_task_top1_labels = []
for key in ['PNâ†’Picture Flashing', 'PNâ†’Auditory Naming', 'PNâ†’Auditory Repetition']:
    if key in cross_task_results:
        pn_cross_task_top1.append(cross_task_results[key]['top_k_accuracies'][1][:min_n_bins])
        pn_cross_task_top1_labels.append(key)

# Plot PF-trained regressors - Top-1
print("\nPlotting PF-trained regressors (Top-1)...")
fig_pf_top1, ax_pf_top1 = plot_accuracy_plotly(
    pf_within_top1,
    *pf_cross_task_top1,
    data_std=[pf_within_top1_std] + [None] * len(pf_cross_task_top1),
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PF Peak Regressor: Top-1 Accuracy on All Tasks",
    ylabel="Top-1 Accuracy",
    data_labels=['PFâ†’PF (within-task)'] + pf_cross_task_top1_labels
)
fig_pf_top1.show()

# Plot PN-trained regressors - Top-1
print("Plotting PN-trained regressors (Top-1)...")
fig_pn_top1, ax_pn_top1 = plot_accuracy_plotly(
    pn_within_top1,
    *pn_cross_task_top1,
    data_std=[pn_within_top1_std] + [None] * len(pn_cross_task_top1),
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PN Peak Regressor: Top-1 Accuracy on All Tasks",
    ylabel="Top-1 Accuracy",
    data_labels=['PNâ†’PN (within-task)'] + pn_cross_task_top1_labels
)
fig_pn_top1.show()

# Print peak Top-1 accuracies
print("\n" + "="*80)
print("Peak Top-1 Accuracies:")
print("="*80)
for combo_name in sorted(cross_task_results.keys()):
    result = cross_task_results[combo_name]
    peak_acc = result['top_k_accuracies'][1].max()
    peak_idx = result['top_k_accuracies'][1].argmax()
    peak_time = peak_idx * bin_size / 1000
    print(f"{combo_name:35s}: Acc = {peak_acc:.3f} at t = {peak_time:.2f}s")
print("="*80)

Creating Top-1 accuracy visualization...


KeyError: 'PFâ†’PF (within)'

## Visualization: Top-K Accuracy Comparison

Compare different top-k accuracy thresholds using plot_accuracy_plotly.

In [ ]:
# Prepare Top-5 accuracy data
print("Creating Top-5 accuracy visualization...")

# PF-trained regressors - Top-5 accuracy
pf_within_top5 = cross_task_results['PFâ†’PF (within)']['top_k_accuracies'][5][:min_n_bins]
pf_within_top5_std = cross_task_results['PFâ†’PF (within)']['top_k_std'][5][:min_n_bins]

pf_cross_task_top5 = []
pf_cross_task_top5_labels = []
for key in ['PFâ†’Picture Naming', 'PFâ†’Auditory Naming', 'PFâ†’Auditory Repetition']:
    if key in cross_task_results:
        pf_cross_task_top5.append(cross_task_results[key]['top_k_accuracies'][5][:min_n_bins])
        pf_cross_task_top5_labels.append(key)

# PN-trained regressors - Top-5 accuracy
pn_within_top5 = cross_task_results['PNâ†’PN (within)']['top_k_accuracies'][5][:min_n_bins]
pn_within_top5_std = cross_task_results['PNâ†’PN (within)']['top_k_std'][5][:min_n_bins]

pn_cross_task_top5 = []
pn_cross_task_top5_labels = []
for key in ['PNâ†’Picture Flashing', 'PNâ†’Auditory Naming', 'PNâ†’Auditory Repetition']:
    if key in cross_task_results:
        pn_cross_task_top5.append(cross_task_results[key]['top_k_accuracies'][5][:min_n_bins])
        pn_cross_task_top5_labels.append(key)

# Plot PF-trained regressors - Top-5
print("\nPlotting PF-trained regressors (Top-5)...")
fig_pf_top5, ax_pf_top5 = plot_accuracy_plotly(
    pf_within_top5,
    *pf_cross_task_top5,
    data_std=[pf_within_top5_std] + [None] * len(pf_cross_task_top5),
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PF Peak Regressor: Top-5 Accuracy on All Tasks",
    ylabel="Top-5 Accuracy",
    data_labels=['PFâ†’PF (within-task)'] + pf_cross_task_top5_labels
)
fig_pf_top5.show()

# Plot PN-trained regressors - Top-5
print("Plotting PN-trained regressors (Top-5)...")
fig_pn_top5, ax_pn_top5 = plot_accuracy_plotly(
    pn_within_top5,
    *pn_cross_task_top5,
    data_std=[pn_within_top5_std] + [None] * len(pn_cross_task_top5),
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PN Peak Regressor: Top-5 Accuracy on All Tasks",
    ylabel="Top-5 Accuracy",
    data_labels=['PNâ†’PN (within-task)'] + pn_cross_task_top5_labels
)
fig_pn_top5.show()

# Print peak Top-5 accuracies
print("\n" + "="*80)
print("Peak Top-5 Accuracies:")
print("="*80)
for combo_name in sorted(cross_task_results.keys()):
    result = cross_task_results[combo_name]
    peak_acc = result['top_k_accuracies'][5].max()
    peak_idx = result['top_k_accuracies'][5].argmax()
    peak_time = peak_idx * bin_size / 1000
    print(f"{combo_name:35s}: Acc = {peak_acc:.3f} at t = {peak_time:.2f}s")
print("="*80)

# Plot comparison of different k values for within-task performance
print("\nCreating Top-K comparison for within-task performance...")

# PF within-task: compare all k values
pf_topk_data = [cross_task_results['PFâ†’PF (within)']['top_k_accuracies'][k][:min_n_bins] for k in top_k_values]
pf_topk_labels = [f'Top-{k}' for k in top_k_values]

fig_pf_topk, ax_pf_topk = plot_accuracy_plotly(
    pf_topk_data[0],
    *pf_topk_data[1:],
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PFâ†’PF (within-task): Comparison of Top-K Accuracies",
    ylabel="Accuracy",
    data_labels=pf_topk_labels
)
fig_pf_topk.show()

# PN within-task: compare all k values
pn_topk_data = [cross_task_results['PNâ†’PN (within)']['top_k_accuracies'][k][:min_n_bins] for k in top_k_values]
pn_topk_labels = [f'Top-{k}' for k in top_k_values]

fig_pn_topk, ax_pn_topk = plot_accuracy_plotly(
    pn_topk_data[0],
    *pn_topk_data[1:],
    back=back,
    forward=forward,
    lines=[0],
    line_labels=['Trial Onset'],
    tick_interval=0.5,
    title="PNâ†’PN (within-task): Comparison of Top-K Accuracies",
    ylabel="Accuracy",
    data_labels=pn_topk_labels
)
fig_pn_topk.show()
            line=dict(width=2),
            hovertemplate=f'<b>Top-{k}</b><br>Time: %{{x:.2f}}s<br>Acc: %{{y:.3f}}<extra></extra>'
        ))
    
    fig_k_comparison.add_vline(x=0, line_dash="dot", line_color="gray", annotation_text="Trial Onset")
    
    fig_k_comparison.update_layout(
        title={
            'text': f"Top-K Accuracy Comparison: {combo_name}",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 18}
        },
        xaxis_title="Time (s)",
        yaxis_title="Accuracy",
        height=500,
        width=1000,
        hovermode='x unified',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor="rgba(255,255,255,0.8)"
        ),
        template="plotly_white"
    )
    
    fig_k_comparison.show()

Top-k accuracy visualization code (uncomment and run with actual data)


## Summary Statistics

Display comprehensive summary of all cross-task regression results.

In [ ]:
print("=" * 80)
print("CROSS-TASK REGRESSION SUMMARY")
print("=" * 80)

# Create summary dataframe
summary_data = []

for combo_name, result in sorted(cross_task_results.items()):
    train_task, test_task = combo_name.split('â†’') if 'â†’' in combo_name else (combo_name, combo_name)
    
    is_within_task = train_task in test_task or test_task in train_task
    
    summary_data.append({
        'Train Task': train_task,
        'Test Task': test_task,
        'Type': 'Within-Task' if is_within_task else 'Cross-Task',
        'Peak RÂ²': f"{result['r2_scores'].max():.3f}",
        'Mean RÂ²': f"{result['r2_scores'].mean():.3f}",
        'Peak Top-1': f"{result['top_k_accuracies'][1].max():.3f}",
        'Peak Top-5': f"{result['top_k_accuracies'][5].max():.3f}",
        'Peak Top-10': f"{result['top_k_accuracies'][10].max():.3f}",
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Compute generalization metrics
print("\n" + "=" * 80)
print("GENERALIZATION ANALYSIS")
print("=" * 80)

# Within-task baselines
within_task_r2 = {}
within_task_top1 = {}
within_task_top5 = {}

for combo_name, result in cross_task_results.items():
    if 'â†’' in combo_name:
        train_task, test_task = combo_name.split('â†’')
        if train_task in test_task or test_task in train_task:
            within_task_r2[train_task] = result['r2_scores'].max()
            within_task_top1[train_task] = result['top_k_accuracies'][1].max()
            within_task_top5[train_task] = result['top_k_accuracies'][5].max()

print("\nWithin-Task Performance (Baseline):")
for task, r2 in sorted(within_task_r2.items()):
    print(f"  {task:20s}: RÂ² = {r2:.3f}, Top-1 = {within_task_top1[task]:.3f}, Top-5 = {within_task_top5[task]:.3f}")

# Cross-task generalization ratios
print("\nCross-Task Generalization (vs. Within-Task Baseline):")
for combo_name, result in sorted(cross_task_results.items()):
    if 'â†’' in combo_name:
        train_task, test_task = combo_name.split('â†’')
        if train_task not in test_task and test_task not in train_task:
            if train_task in within_task_r2:
                r2_ratio = result['r2_scores'].max() / within_task_r2[train_task]
                top1_ratio = result['top_k_accuracies'][1].max() / within_task_top1[train_task]
                top5_ratio = result['top_k_accuracies'][5].max() / within_task_top5[train_task]
                
                print(f"\n  {combo_name}:")
                print(f"    RÂ² ratio: {r2_ratio:.2%} (cross/within)")
                print(f"    Top-1 ratio: {top1_ratio:.2%}")
                print(f"    Top-5 ratio: {top5_ratio:.2%}")

# Analysis by modality
print("\n" + "=" * 80)
print("MODALITY ANALYSIS")
print("=" * 80)

visual_to_visual = []
visual_to_auditory = []
auditory_to_visual = []
auditory_to_auditory = []

for combo_name, result in cross_task_results.items():
    if 'â†’' in combo_name:
        train_task, test_task = combo_name.split('â†’')
        peak_r2 = result['r2_scores'].max()
        
        train_is_visual = 'Picture' in train_task or 'Flashing' in train_task
        test_is_visual = 'Picture' in test_task or 'Flashing' in test_task
        
        if train_is_visual and test_is_visual:
            visual_to_visual.append((combo_name, peak_r2))
        elif train_is_visual and not test_is_visual:
            visual_to_auditory.append((combo_name, peak_r2))
        elif not train_is_visual and test_is_visual:
            auditory_to_visual.append((combo_name, peak_r2))
        else:
            auditory_to_auditory.append((combo_name, peak_r2))

print("\nVisual â†’ Visual:")
for combo, r2 in visual_to_visual:
    print(f"  {combo:40s}: RÂ² = {r2:.3f}")

print("\nVisual â†’ Auditory:")
for combo, r2 in visual_to_auditory:
    print(f"  {combo:40s}: RÂ² = {r2:.3f}")

print("\nAuditory â†’ Visual:")
for combo, r2 in auditory_to_visual:
    print(f"  {combo:40s}: RÂ² = {r2:.3f}")

print("\nAuditory â†’ Auditory:")
for combo, r2 in auditory_to_auditory:
    print(f"  {combo:40s}: RÂ² = {r2:.3f}")

print("\n" + "=" * 80)
print(f"Analysis complete! Tested {len(cross_task_results)} trainâ†’test combinations")
print("=" * 80)

CROSS-TASK REGRESSION SUMMARY

1. Picture Flashing (train & test):
   Peak RÂ² score: 0.013
   Mean RÂ² score: -0.033
   Peak ranked accuracy (top-1): 0.001
   Peak top-5 accuracy: 0.007

2. Picture Naming (train & test):
   Peak RÂ² score: -0.478
   Mean RÂ² score: -0.628
   Peak ranked accuracy (top-1): 0.018
   Peak top-5 accuracy: 0.069

3. Cross-task (PF train â†’ PN test):
   Peak RÂ² score: 0.012
   Mean RÂ² score: -0.038
   Peak ranked accuracy (top-1): 0.026
   Peak top-5 accuracy: 0.094

Generalization ratios (cross-task / within-task PF):
   RÂ² score: 93.02%
   Ranked accuracy: 1816.42%

Regression parameters used:
   Regressor: Ridge (alpha=1.0)
   X dimensionality reduction: PCA (50 components)
   Y dimensionality reduction: None
   Bins of history: 1
   Training epochs: 20
   K-folds: 5
   Top-k values: [1, 3, 5, 10, 20]
Summary statistics code (uncomment and run with actual data)


## Analysis Recommendations

### For Auditory Naming Tasks:

**Why time warping is important:**
- Auditory naming has much longer stimulus presentation (~2-3s vs ~0.5s for visual tasks)
- Semantic processing happens at later stage during the longer presentation
- Time warping aligns comparable cognitive stages (e.g., go cue â†’ voice onset) across tasks

**Recommended analyses:**

1. **Within-Task (Original Timing):**
   ```python
   alignment_method = 'keep_original'
   # Analyze each task separately with its native timing
   # Good for understanding task-specific dynamics
   ```

2. **Cross-Task Comparison (Time Warped):**
   ```python
   alignment_method = 'time_warp'
   reference_task = 'picture_naming'
   # Compare semantic representations across tasks
   # Good for understanding shared representations
   ```

3. **Both Analyses:**
   - Train on warped auditory data â†’ test on visual data
   - Train on original auditory data â†’ test on warped visual data
   - Compare to understand impact of temporal dynamics

### Expected Findings:

- **Time warping should improve cross-task generalization** because it aligns comparable cognitive stages
- **Original timing better captures native task dynamics** but may show poor cross-task transfer
- **Ranked accuracy more interpretable than RÂ²** for understanding semantic prediction quality

### Task-Specific Considerations:

- **Auditory Naming**: Longest presentation, semantic processing delayed
- **Auditory Repetition**: Shorter than naming, more motor-focused
- **Picture Naming**: Medium timing, good reference for warping
- **Picture Flashing**: Shortest, purest visual response